# 🛡️ SafeNet AI — Hate Speech Detection
## End-to-End NLP Capstone Project
**Python 3.14 Compatible**

### Pipeline
Dataset → Preprocessing → TF-IDF → Model Training → Evaluation → Visualisation


In [ ]:
# Install dependencies
import subprocess, sys
pkgs = ['pandas','numpy','scikit-learn','nltk','matplotlib','seaborn','wordcloud','flask']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('All packages installed.')

In [ ]:
from __future__ import annotations
import re, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

LABEL_MAP = {0: 'Hate Speech', 1: 'Offensive', 2: 'Neutral'}
print('Libraries loaded OK')

## 1. Load Dataset

In [ ]:
# Load from file or generate synthetic data
from pathlib import Path

DATA_PATH = Path('data/dataset.csv')
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print('Loaded from file:', df.shape)
else:
    print('dataset.csv not found — generating synthetic demo data')
    rng = np.random.default_rng(42)
    hate = ['I hate those people they should all be removed'] * 150
    off  = ['You are a complete idiot stop posting nonsense'] * 150
    neut = ['I had a wonderful day walking in the park today'] * 150
    tweets = hate + off + neut
    labels = [0]*150 + [1]*150 + [2]*150
    df = pd.DataFrame({'tweet': tweets, 'label': labels})
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.head())
print('\nShape:', df.shape)
print('\nLabel distribution:')
print(df['label'].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
label_names = [LABEL_MAP[i] for i in sorted(df['label'].unique())]
counts = df['label'].value_counts().sort_index()
colors = ['#e74c3c', '#f39c12', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(label_names, counts.values, color=colors)
axes[0].set_title('Class Distribution — Bar Chart')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=label_names, colors=colors, autopct='%1.1f%%')
axes[1].set_title('Class Distribution — Pie Chart')
plt.tight_layout()
plt.show()

In [ ]:
# Text length distribution
df['text_len'] = df['tweet'].str.len()
fig, ax = plt.subplots(figsize=(10, 4))
for i, (lid, lname) in enumerate(LABEL_MAP.items()):
    subset = df[df['label'] == lid]['text_len']
    ax.hist(subset, bins=30, alpha=0.6, label=lname, color=colors[i])
ax.set_title('Text Length Distribution by Class')
ax.set_xlabel('Character Count')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Text Preprocessing

In [ ]:
for resource in ('punkt', 'stopwords', 'wordnet', 'punkt_tab'):
    nltk.download(resource, quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

lemmatiser = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def lemmatise(text: str) -> str:
    try:
        tokens = word_tokenize(text)
    except Exception:
        tokens = text.split()
    tokens = [lemmatiser.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

def preprocess(text: str) -> str:
    return lemmatise(clean_text(text))

df['clean_text'] = df['tweet'].apply(preprocess)

print('Before:', df['tweet'].iloc[0])
print('After: ', df['clean_text'].iloc[0])

## 4. Word Cloud

In [ ]:
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    cmaps = ['Reds', 'Oranges', 'Greens']
    for i, (lid, lname) in enumerate(LABEL_MAP.items()):
        texts = ' '.join(df[df['label'] == lid]['clean_text'])
        if texts.strip():
            wc = WordCloud(width=400, height=200, background_color='white',
                           colormap=cmaps[i], max_words=60).generate(texts)
            axes[i].imshow(wc, interpolation='bilinear')
        axes[i].axis('off')
        axes[i].set_title(lname)
    plt.suptitle('Word Clouds by Class', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('pip install wordcloud to see word clouds')

## 5. TF-IDF Vectorisation & Train/Test Split

In [ ]:
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df['clean_text'], df['label'],
    test_size=0.20, random_state=42, stratify=df['label']
)

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vectorizer.fit_transform(X_train_txt)
X_test  = vectorizer.transform(X_test_txt)

print('Train shape:', X_train.shape)
print('Test  shape:', X_test.shape)

## 6. Train & Compare All Models

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes':         MultinomialNB(alpha=0.5),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (LinearSVC)':     LinearSVC(max_iter=2000, random_state=42),
}

results = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds, average='weighted', zero_division=0), 4),
        'Recall':    round(recall_score(y_test, preds, average='weighted', zero_division=0), 4),
        'F1-Score':  round(f1_score(y_test, preds, average='weighted', zero_division=0), 4),
    }
    print(name, '->', results[name])

res_df = pd.DataFrame(results).T
print('\nFull comparison table:')
res_df

## 7. Model Comparison Chart

In [ ]:
metrics     = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_names = list(results.keys())
x      = np.arange(len(model_names))
width  = 0.2
bar_colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (metric, color) in enumerate(zip(metrics, bar_colors)):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(x + i * width, vals, width, label=metric, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                str(round(bar.get_height(), 3)),
                ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, fontsize=9)
ax.set_ylim(0, 1.13)
ax.set_title('Model Comparison — All Metrics', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Confusion Matrix — Best Model

In [ ]:
# Pick best model by F1-Score
best_name = max(results, key=lambda m: results[m]['F1-Score'])
print('Best model:', best_name)

best_clf = models[best_name]
preds    = best_clf.predict(X_test)
cm       = confusion_matrix(y_test, preds)
names    = [LABEL_MAP[i] for i in sorted(LABEL_MAP)]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=names, yticklabels=names, ax=ax)
ax.set_title(best_name + ' — Confusion Matrix', fontsize=13, fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, preds, target_names=names, zero_division=0))

## 9. Real-Time Prediction Test

In [ ]:
test_texts = [
    'I hate all those people they should be removed',
    'You are such an idiot stop talking nonsense',
    'Today was a great day I went to the park',
    'Those disgusting animals do not belong here',
    'Just had a coffee with a good friend',
]

print('Real-Time Predictions\n' + '='*55)
for text in test_texts:
    cleaned = preprocess(text)
    feat    = vectorizer.transform([cleaned])
    pred    = int(best_clf.predict(feat)[0])
    label   = LABEL_MAP[pred]
    emoji   = {'Hate Speech': '🚨', 'Offensive': '⚠️', 'Neutral': '✅'}[label]
    print(emoji, label.ljust(14), '|', text[:55])

## 10. Save Model

In [ ]:
from pathlib import Path
Path('models').mkdir(exist_ok=True)

with open('models/best_model.pkl', 'wb') as f:
    pickle.dump(best_clf, f)
with open('models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

meta = {
    'best_model': best_name,
    'label_map': {str(k): v for k, v in LABEL_MAP.items()},
    'results': {m: {k: float(v) for k, v in r.items()} for m, r in results.items()},
}
with open('models/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

print('Model saved to models/')
print('Best model:', best_name)
print('F1-Score:', results[best_name]['F1-Score'])